In [14]:
pip list 

Package                                  Version
---------------------------------------- -----------
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.3
aiosignal                                1.4.0
anaconda-anon-usage                      0.7.5
anaconda-auth                            0.12.3
anaconda-cli-base                        0.7.0
annotated-types                          0.6.0
anyio                                    4.12.1
archspec                                 0.2.5
argon2-cffi                              25.1.0
argon2-cffi-bindings                     25.1.0
arrow                                    1.4.0
asttokens                                3.0.1
async-lru                                2.2.0
attrs                                    25.4.0
babel                                    2.18.0
backoff                                  2.2.1
bcrypt                                   5.0.0
beautifulsoup4                           4.14

In [1]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from openai import OpenAI

import requests


from langchain_tavily import TavilySearch

In [2]:
gpt_4o = ChatOpenAI(model='gpt-4o-mini',
        api_key="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",
        base_url='https://openrouter.ai/api/v1')



In [5]:
prompt1 = ChatPromptTemplate.from_template('Explain the following topin in 50 Words : {topic}')

In [6]:
prompt1.format_messages(topic='Messi')


[HumanMessage(content='Explain the following topin in 50 Words : Messi', additional_kwargs={}, response_metadata={})]

In [7]:
result = gpt_4o.invoke(prompt1.format_messages(topic='Messi'))

In [8]:
result.content

"Lionel Messi, an Argentine professional footballer, is widely regarded as one of the greatest players in history. Known for his incredible dribbling, vision, and goal-scoring ability, he spent the majority of his career at FC Barcelona, winning numerous titles, including multiple Ballon d'Or awards, before joining Paris Saint-Germain in 2021."

In [12]:
def get_result(topic:str):
    prompt = ChatPromptTemplate.from_template("Explain the following topic in 50 Words : {topic}" )
    result = gpt_4o.invoke(prompt.format_messages(topic=topic)).content
    return result


def get_result_by_chain(topic:str):
    prompt = ChatPromptTemplate.from_template("Explain the following topic in 50 Words : {topic}" )
    chain = prompt | gpt_4o | StrOutputParser() 
    return chain.invoke(input={'topic':topic})
    

In [47]:
get_result('CR7')

"CR7, the nickname for Cristiano Ronaldo, is a renowned professional footballer from Portugal, celebrated for his exceptional skills, athleticism, and goal-scoring ability. With multiple Ballon d'Or awards, he has played for clubs like Manchester United, Real Madrid, and Juventus, and is regarded as one of the greatest players in the sport's history."

In [13]:
get_result_by_chain('messi')

"Lionel Messi, an Argentine professional footballer, is widely regarded as one of the greatest players in history. Known for his exceptional dribbling, vision, and goal-scoring ability, he spent the majority of his career at FC Barcelona, winning numerous titles, including multiple Ballon d'Or awards, before joining Paris Saint-Germain in 2021."

In [175]:
@tool
def add(a:int,b:int)->int:
    """
    args: a and b are 2 params
    
    This function add two numbers 
    and return their sum
    """
    return a + b

@tool
def multiply(a:int,b:int)->int:
    """
    args: a and b are 2 params
    
    This function multiply two numbers 
    and return their product
    """
    return a * b
    
    

In [176]:
math_tool=[add,multiply]

In [177]:
math_tool

[StructuredTool(name='add', description='args: a and b are 2 params\n\nThis function add two numbers \nand return their sum', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x000002B27434D760>),
 StructuredTool(name='multiply', description='args: a and b are 2 params\n\nThis function multiply two numbers \nand return their product', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x000002B2741BA520>)]

In [178]:
system_promt = """
You are a helpful Math Assistant
Always use tools for calculation
"""


In [179]:
agent1 = create_agent(gpt_4o,tools=math_tool,system_prompt=system_promt)

In [39]:
result = agent1.invoke({'messages':'Pls add 4 & 5'})

In [41]:
result['messages']

[HumanMessage(content='Pls add 4 & 5', additional_kwargs={}, response_metadata={}, id='b11e548a-edcd-4dbd-a041-5cb307190d23'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 125, 'total_tokens': 142, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 2.895e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.895e-05, 'upstream_inference_prompt_cost': 1.875e-05, 'upstream_inference_completions_cost': 1.02e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_2d82c05f26', 'id': 'gen-1772384889-SSjMKfSkww1KG9bGzumG', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019caa5f-2866-7380-874f-3aade8819e3a-0', tool_calls=[{'name': 'add', 'args': {'a': 4, 'b': 5}, 'id': 

In [42]:
len(result['messages'])

4

In [46]:
result['messages'][-2]

ToolMessage(content='9', name='add', id='93c2235f-a1a0-4870-be93-26e689e31243', tool_call_id='call_cuDJG8aUS4cgf2gQZmsXCSo5')

In [43]:
result['messages'][-1]

AIMessage(content='The sum of 4 and 5 is 9.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 150, 'total_tokens': 163, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 3.03e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 3.03e-05, 'upstream_inference_prompt_cost': 2.25e-05, 'upstream_inference_completions_cost': 7.8e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_2d82c05f26', 'id': 'gen-1772384890-o8UJ9T4oYx0Qfbse7GQC', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019caa5f-2cf2-7e00-9186-d7f59bda5cd3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 150, 'output_tokens': 13, 'total_tokens': 163, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'ou

In [99]:
class reply(BaseModel):
    Answer : str
    
    

In [180]:
agent2 = create_agent(gpt_4o,tools=math_tool,system_prompt=system_promt,response_format=reply)

In [181]:
result2 = agent2.invoke(
              {'messages':
              [
                  {'role':'user','content':'who is messi'}]
             })

In [102]:
result2

{'messages': [HumanMessage(content='who is messi', additional_kwargs={}, response_metadata={}, id='ebd13396-4c43-435c-9213-7f394e1696b0'),
  AIMessage(content='{"Answer":"Lionel Messi, full name Lionel Andrés Messi, is an Argentine professional footballer widely regarded as one of the greatest players of all time. Born on June 24, 1987, in Rosario, Argentina, he began playing football at a young age and joined FC Barcelona\'s youth academy, La Masia, at 13. \\n\\nMessi made his professional debut for Barcelona in 2004 and went on to spend over 20 years with the club, during which he won numerous trophies, including multiple UEFA Champions League titles and La Liga championships. He is known for his extraordinary dribbling, playmaking ability, and prolific goal-scoring.\\n\\nIn August 2021, Messi joined Paris Saint-Germain (PSG) after leaving Barcelona due to financial constraints faced by the club. He has also had significant success with the Argentina national team, winning the Copa A

In [64]:
dict(result2)

{'messages': [HumanMessage(content='Add 3 and 4', additional_kwargs={}, response_metadata={}, id='bb2ae627-04dd-4789-a47b-ae69ed9a5d11'),
  AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 149, 'total_tokens': 166, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 3.255e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 3.255e-05, 'upstream_inference_prompt_cost': 2.235e-05, 'upstream_inference_completions_cost': 1.02e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_373a14eb6f', 'id': 'gen-1772385559-A8LHwy3tP8qETZ3CdD3Y', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019caa69-621f-7e61-9a9c-e270f98b9e23-0', tool_calls=[{'name': 'add', 'arg

In [182]:
result2['structured_response']

reply(Answer="Lionel Messi, born on June 24, 1987, in Rosario, Argentina, is a professional football (soccer) player widely regarded as one of the greatest players of all time. He began his career at FC Barcelona, where he spent over 20 years and became the club's all-time leading scorer. During his time at Barcelona, he won numerous domestic league titles, UEFA Champions League titles, and individual awards, including multiple FIFA Ballon d'Or trophies, awarded to the world's best player.\n\nIn 2021, Messi transferred to Paris Saint-Germain (PSG) after financial constraints prevented Barcelona from renewing his contract. He has also had a successful international career with the Argentina national team, winning the Copa America in 2021 and the FIFA World Cup in 2022. Known for his incredible dribbling, vision, and scoring ability, Messi has left a profound legacy in the world of football.")

In [121]:
soup.find_all?

Signature:
soup.find_all(
    name: '_FindMethodName' = None,
    attrs: 'Optional[_StrainableAttributes]' = None,
    recursive: 'bool' = True,
    string: 'Optional[_StrainableString]' = None,
    limit: 'Optional[int]' = None,
    _stacklevel: 'int' = 2,
    **kwargs: '_StrainableAttribute',
) -> 'Union[_SomeTags, _SomeNavigableStrings, _QueryResults]'
Docstring:
Look in the children of this `PageElement` and find all
`PageElement` objects that match the given criteria.

All find_* methods take a common set of arguments. See the online
documentation for detailed explanations.

:param name: A filter on tag name.
:param attrs: Additional filters on attribute values.
:param recursive: If this is True, find_all() will perform a
    recursive search of this PageElement's children. Otherwise,
    only the direct children will be considered.
:param limit: Stop looking after finding this many results.
:param _stacklevel: Used internally to improve warning messages.
:kwargs: Additional filte

In [137]:
for genre in soup.find_all('li'):
    print(genre.text)

thriller book
mystery book>


In [142]:
soup.find_all('ul')

[<ul class="book">
 <li>thriller book</li>
 <li>mystery book</li>
 </ul>,
 <ul class="soccer">
 <li>Messi</li>
 <li>CR7</li>
 </ul>]

In [143]:
soup.find_all?

Signature:
soup.find_all(
    name: '_FindMethodName' = None,
    attrs: 'Optional[_StrainableAttributes]' = None,
    recursive: 'bool' = True,
    string: 'Optional[_StrainableString]' = None,
    limit: 'Optional[int]' = None,
    _stacklevel: 'int' = 2,
    **kwargs: '_StrainableAttribute',
) -> 'Union[_SomeTags, _SomeNavigableStrings, _QueryResults]'
Docstring:
Look in the children of this `PageElement` and find all
`PageElement` objects that match the given criteria.

All find_* methods take a common set of arguments. See the online
documentation for detailed explanations.

:param name: A filter on tag name.
:param attrs: Additional filters on attribute values.
:param recursive: If this is True, find_all() will perform a
    recursive search of this PageElement's children. Otherwise,
    only the direct children will be considered.
:param limit: Stop looking after finding this many results.
:param _stacklevel: Used internally to improve warning messages.
:kwargs: Additional filte

In [144]:
soup.find_all('ul',attrs={'class':'soccer'})

[<ul class="soccer">
 <li>Messi</li>
 <li>CR7</li>
 </ul>]

In [146]:
import requests

In [161]:
url="https://books.toscrape.com/catalogue/page-{}.html"
r = requests.get("https://books.toscrape.com/catalogue/category/books_1/index.html")

In [157]:
url

'https://books.toscrape.com/catalogue/category/books_1/index.html'

In [183]:
# tools/fetch_html.py

from langchain.tools import tool
import requests

@tool
def fetch_html(url: str) -> str:
    """Fetch HTML content from a website."""

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers, timeout=15)

    return response.text

In [187]:
agent3 = create_agent(gpt_4o,system_prompt=scrap_prompt,response_format=reply)

In [193]:
result2 = agent3.invoke(
              {'messages':
              [
                  {'role':'user','content':'http://sbi.com/'}]
             })

In [194]:
result2

{'messages': [HumanMessage(content='http://sbi.com/', additional_kwargs={}, response_metadata={}, id='b41483aa-3026-423e-9ad0-3bfabd78b5b2'),
  AIMessage(content='It looks like you’ve provided a link to what might be the State Bank of India (SBI) website, but the URL seems to be incomplete or incorrect. The official website for the State Bank of India is typically https://www.sbi.co.in. If you need specific information about SBI, such as banking services, loans, or customer service, feel free to ask!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 22, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 4.89e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 4.89e-05, 'upstream_inference_prompt_cost': 3.3e-06, 'upstream_

In [25]:
from pydantic import BaseModel
from typing import List

class ScrapedData(BaseModel):
    items: List[str]

In [3]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key="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"
)

In [34]:
@tool
def fetch_html(url: str) -> str:
    """Fetch raw HTML from a webpage."""
    response = requests.get(url, timeout=15)
    return response.text[:1000]

In [35]:
@tool
def generate_xpath(html:str,target:str)->str:
    
    """
    Generate XPath for target element from HTML.
    target example: product name, price, title
    """

    prompt = f"""
      Find XPath for {target} in this HTML:
    
      {html}
    
      Return only XPath.
              """

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content


@tool
def extract_with_xpath(html_content: str, xpath: str) -> list:
    """Extract data using XPath."""

    tree = html.fromstring(html_content)

    results = tree.xpath(xpath)

    return [str(r).strip() for r in results]


In [32]:
"""
agent4 = create_agent(model=gpt_4o,tools=[fetch_html,generate_xpath],
                      system_prompt="""
                                    You are a web scraping agent.

                                    Steps:
                                    1. fetch_html
                                    2. generate_xpath
                                    
                                    Never hallucinate.
                                    """,
                      response_format=ScrapedData
                     )
"""
                      

In [33]:
"""
result4 = agent4.invoke({
              'messages':[
              {'role':'user','content':'URL is https://books.toscrape.com/ '}
              ]
              } 
              )
"""

"\nresult4 = agent4.invoke({\n              'messages':[\n              {'role':'user','content':'URL is https://books.toscrape.com/ '}\n              ]\n              } \n              )\n"

In [24]:
result4['messages'][-1].content

"I have fetched the HTML content from the specified URL. Now, I will proceed to generate the XPath expressions for various elements on the page. This will help in extracting specific data from the HTML.\n\nHere are the XPath expressions for key elements:\n\n1. **Title of the page**: \n   ```xpath\n   //title/text()\n   ```\n\n2. **Product Titles**: \n   ```xpath\n   //h3/a/text()\n   ```\n\n3. **Product Links**: \n   ```xpath\n   //h3/a/@href\n   ```\n\n4. **Product Prices**: \n   ```xpath\n   //p[@class='price_color']/text()\n   ```\n\n5. **Product Availability**: \n   ```xpath\n   //p[@class='instock availability']/text()\n   ```\n\n6. **Star Ratings**: \n   ```xpath\n   //p[contains(@class, 'star-rating')]/@class\n   ```\n\n7. **All Categories**:\n   ```xpath\n   //ul[@class='nav nav-list']//li/a/text()\n   ```\n\nThese XPath expressions can be used to extract specific elements from the HTML of the page you provided. If you need further assistance or specific data extraction, please

In [43]:
class Expence(BaseModel):
    bill:int
    

In [3]:
@tool
def get_HR(city:str)->int:
    """
    Args : city must be an string
    This function return Housing Rent
    """
    
    if city == 'Delhi':
        return 4000
    elif city == 'Mumbai':
        return 10000
    elif city=='Chennai':
        return 8000
    else:
        return 6000
        
        

def get_GrossaryBill(food:str)->int:
    """
    Args :food must be an string
    This function return Grossary Bill
    
    """
    if food == 'Vegetarian':
        return 10000
    elif food == "Non Vegetaria":
        return 15000
    

In [53]:
budget_prompt="""
        You are a budget assistant 
        Your job is to calculate monthly budget using tools defined
        
        Steps :
        1. get_HR
        2. get_GrossaryBill
        """

In [54]:
budget = create_agent(gpt_4o,tools=[get_HR,get_GrossaryBill],system_prompt=budget_prompt,response_format=Expence)

In [55]:
result5 = budget.invoke({
              'messages':[
              {'role':'user','content':'I am a Vegetarian live in Delhi '}
              ]
              } 
              )

In [56]:
result5

{'messages': [HumanMessage(content='I am a Vegetarian live in Delhi ', additional_kwargs={}, response_metadata={}, id='6ec32cb1-2426-47ae-b3d1-1bbcfa1004ec'),
  AIMessage(content='', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 163, 'total_tokens': 212, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 5.385e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 5.385e-05, 'upstream_inference_prompt_cost': 2.445e-05, 'upstream_inference_completions_cost': 2.94e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_ff0ad3d6fd', 'id': 'gen-1772399120-hrhH6eOv9D8xK8YTfov3', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cab38-4d85-7460-8381-426f53345a50-0', tool_calls=

In [57]:
result5['structured_response']

Expence(bill=14000)

In [105]:
from TypedDict import List

ModuleNotFoundError: No module named 'TypedDict'

In [30]:
# @tool
def get_day(Name:str)->str:
    """
    Args : A Dictionary name GF contain 7 keys
    Each key map to a list. This list contain
    Day Name, City Name, Food Choice
    This function return name of day 
    """
    GF = {
            'Neha'   : ['Monday','Delhi','Japanese'],
            'Sonal'  : ['Tuesday','Mumbai','Italian'],
            'Monica' : ['Wednesday','Chennai','Chinese'],
            'Venus'  : ['Thrusday','Bangalore','Indian'],
            'Veronica' :  ['Friday','Dubai','Mexican'],
            'Rashmi'   :  ['Saturday','London','Korean'],
            'Wife' : ['Sunday','Delhi','Home Made']
        }

    if Name in GF.keys():
        return GF[Name]
    else:
        raise KeyError(f'This {Name} not found in our list')

@tool
def find_expense(info):
    """
    Args : info
    This fuction return flight information along with price
    """
    
   
    search = TavilySearch(max_results=2,topic="general")
    
    tavily_result = search.invoke(info)

    for result in tavily_result['results']:
        print(result['content'])
   

In [31]:
agent6 = create_agent(gpt_4o,tools=[find_expense],system_prompt="You are an affordable travel planner")

In [4]:
search = TavilySearch(
       max_results=2,
       topic="general",
    )

In [47]:
travel_prompt="""
Travel Date = March 06, 2026
from city = Delhi
to city = Mumbai
No of Passengers = 1

"""

In [83]:
result6 = agent6.invoke({
                'messages':prompt
                        }
             )

In [84]:
result6['messages'][-1]

AIMessage(content="To create a detailed travel plan for your trip to Kerala in April 2026, I'll need to gather some information about your preferences:\n\n1. **Departure City:** Where will you be flying from?\n2. **Duration of Stay:** How long do you plan to stay in Kerala?\n3. **Interests:** Are you interested in specific activities like sightseeing, beach activities, cultural experiences, etc.?\n4. **Budget:** Do you have a budget range for the trip (excluding flight)?\n5. **Traveling Companions:** Are you traveling alone or with others? If so, how many?\n\nWith this information, I'll be able to provide you with a more tailored travel plan and cost estimates.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 143, 'prompt_tokens': 90, 'total_tokens': 233, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0

In [73]:
travel_prompt = ChatPromptTemplate.from_messages([("system","You are a travel planner"),
                                                   ("user","I am planning to go Kerala in April 2026 could you please tell me travel plan along with cost")
                                                    ])  

In [74]:
travel_prompt

ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a travel planner'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='I am planning to go Kerala in April 2026 could you please tell me travel plan along with cost'), additional_kwargs={})])

In [77]:
from langchain_core.messages import HumanMessage,SystemMessage

In [78]:
travel_prompt = ChatPromptTemplate.from_messages([("system","You are a travel planner"),
                                                   ("user","I am planning to go Kerala in April 2026 could you please tell me travel plan along with cost")
                                                    ])  

In [81]:
prompt =[SystemMessage("You are a travel planner"),HumanMessage("I am planning to go Kerala in April 2026 could you please tell me travel plan along with cost")]

In [82]:
prompt

[SystemMessage(content='You are a travel planner', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I am planning to go Kerala in April 2026 could you please tell me travel plan along with cost', additional_kwargs={}, response_metadata={})]